In [0]:
from pyspark.sql import Row
from datetime import date, timedelta
import random

random.seed(42)

# -----------------------------
# Synthetic Products
# -----------------------------
products = [
    Row(product_id="P001", product_name="Tomatoes", category="Vegetable"),
    Row(product_id="P002", product_name="Cucumbers", category="Vegetable"),
    Row(product_id="P003", product_name="Peppers", category="Vegetable"),
    Row(product_id="P004", product_name="Strawberries", category="Fruit"),
    Row(product_id="P005", product_name="Lettuce", category="Vegetable"),
]

df_products = spark.createDataFrame(products)

# -----------------------------
# Synthetic Shipments
# -----------------------------
shipments = []

start_date = date(2026, 1, 1)

for i in range(1, 501):
    shipment_id = f"S{i:04d}"
    product_id = random.choice(["P001", "P002", "P003", "P004", "P005"])
    shipment_date = start_date + timedelta(days=random.randint(0, 250))
    shipped_kg = random.randint(100, 1000)

    shipments.append(
        Row(
            shipment_id=shipment_id,
            product_id=product_id,
            shipment_date=shipment_date,
            shipped_kg=float(shipped_kg)
        )
    )

# Controlled data-quality issues
shipments.append(
    Row(
        shipment_id="S0050",
        product_id="P001",
        shipment_date=start_date,
        shipped_kg=500.0
    )
)

shipments.append(
    Row(
        shipment_id=None,
        product_id="P003",
        shipment_date=start_date,
        shipped_kg=300.0
    )
)

shipments.append(
    Row(
        shipment_id="S0501",
        product_id="P002",
        shipment_date=start_date,
        shipped_kg=-50.0
    )
)

df_shipments = spark.createDataFrame(shipments)

# -----------------------------
# Synthetic Receipts
# -----------------------------
receipts = []

for i in range(1, 471):
    shipment_id = f"S{i:04d}"
    receipt_id = f"R{i:04d}"

    shipped_value = next(
        r.shipped_kg for r in shipments
        if r.shipment_id == shipment_id
    )

    # Most receipts match exactly
    received_kg = shipped_value

    # Introduce quantity discrepancies
    if i % 25 == 0:
        received_kg = shipped_value - random.randint(5, 30)

    receipts.append(
        Row(
            receipt_id=receipt_id,
            shipment_id=shipment_id,
            receipt_date=start_date + timedelta(days=random.randint(1, 260)),
            received_kg=float(received_kg)
        )
    )

# Controlled receipt issues
receipts.append(
    Row(
        receipt_id="R0100",
        shipment_id="S0100",
        receipt_date=start_date,
        received_kg=400.0
    )
)

receipts.append(
    Row(
        receipt_id=None,
        shipment_id="S0200",
        receipt_date=start_date,
        received_kg=350.0
    )
)

receipts.append(
    Row(
        receipt_id="R0500",
        shipment_id="S0300",
        receipt_date=start_date,
        received_kg=-10.0
    )
)

df_receipts = spark.createDataFrame(receipts)

print("Products:", df_products.count())
print("Shipments:", df_shipments.count())
print("Receipts:", df_receipts.count())

from pyspark.sql.functions import current_timestamp, lit

# Add Bronze metadata
df_products_bronze = (
    df_products
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("SYNTHETIC_OPERATIONAL"))
)

df_shipments_bronze = (
    df_shipments
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("SYNTHETIC_OPERATIONAL"))
)

df_receipts_bronze = (
    df_receipts
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("SYNTHETIC_OPERATIONAL"))
)

# Save Bronze Delta tables
(
    df_products_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.greenhouse_bronze.raw_products")
)

(
    df_shipments_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.greenhouse_bronze.raw_shipments")
)

(
    df_receipts_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.greenhouse_bronze.raw_receipts")
)

# Verify
print(
    "Bronze products:",
    spark.table("workspace.greenhouse_bronze.raw_products").count()
)

print(
    "Bronze shipments:",
    spark.table("workspace.greenhouse_bronze.raw_shipments").count()
)

print(
    "Bronze receipts:",
    spark.table("workspace.greenhouse_bronze.raw_receipts").count()
)

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --------------------------------------------------
# Read Bronze
# --------------------------------------------------
products_b = spark.table(
    "workspace.greenhouse_bronze.raw_products"
)

shipments_b = spark.table(
    "workspace.greenhouse_bronze.raw_shipments"
)

receipts_b = spark.table(
    "workspace.greenhouse_bronze.raw_receipts"
)

# --------------------------------------------------
# PRODUCTS - all generated product rows are valid
# --------------------------------------------------
products_clean = products_b

# --------------------------------------------------
# SHIPMENTS - validate
# --------------------------------------------------
shipment_window = Window.partitionBy("shipment_id")

shipments_checked = (
    shipments_b
    .withColumn(
        "duplicate_count",
        F.count("*").over(shipment_window)
    )
    .withColumn(
        "validation_reason",
        F.when(
            F.col("shipment_id").isNull(),
            "MISSING_SHIPMENT_ID"
        )
        .when(
            F.col("shipped_kg").isNull(),
            "MISSING_SHIPPED_KG"
        )
        .when(
            F.col("shipped_kg") <= 0,
            "INVALID_SHIPPED_KG"
        )
        .when(
            F.col("duplicate_count") > 1,
            "DUPLICATE_SHIPMENT_ID"
        )
        .otherwise("VALID")
    )
)

shipments_clean = (
    shipments_checked
    .filter(F.col("validation_reason") == "VALID")
    .drop("duplicate_count")
)

shipments_quarantine = (
    shipments_checked
    .filter(F.col("validation_reason") != "VALID")
    .drop("duplicate_count")
)

# --------------------------------------------------
# RECEIPTS - structural validation
# --------------------------------------------------
receipt_window = Window.partitionBy("receipt_id")

receipts_checked = (
    receipts_b
    .withColumn(
        "duplicate_count",
        F.count("*").over(receipt_window)
    )
    .withColumn(
        "base_validation_reason",
        F.when(
            F.col("receipt_id").isNull(),
            "MISSING_RECEIPT_ID"
        )
        .when(
            F.col("shipment_id").isNull(),
            "MISSING_SHIPMENT_ID"
        )
        .when(
            F.col("received_kg").isNull(),
            "MISSING_RECEIVED_KG"
        )
        .when(
            F.col("received_kg") <= 0,
            "INVALID_RECEIVED_KG"
        )
        .when(
            F.col("duplicate_count") > 1,
            "DUPLICATE_RECEIPT_ID"
        )
        .otherwise("VALID")
    )
)

# --------------------------------------------------
# RECEIPTS - referential integrity check
# --------------------------------------------------
valid_shipment_ids = (
    shipments_clean
    .select(
        F.col("shipment_id").alias("valid_shipment_id")
    )
    .distinct()
)

receipts_checked = (
    receipts_checked
    .join(
        valid_shipment_ids,
        receipts_checked.shipment_id ==
        valid_shipment_ids.valid_shipment_id,
        "left"
    )
    .withColumn(
        "validation_reason",
        F.when(
            F.col("base_validation_reason") != "VALID",
            F.col("base_validation_reason")
        )
        .when(
            F.col("valid_shipment_id").isNull(),
            "ORPHAN_SHIPMENT_ID"
        )
        .otherwise("VALID")
    )
    .drop(
        "duplicate_count",
        "base_validation_reason",
        "valid_shipment_id"
    )
)

receipts_clean = (
    receipts_checked
    .filter(F.col("validation_reason") == "VALID")
)

receipts_quarantine = (
    receipts_checked
    .filter(F.col("validation_reason") != "VALID")
)

# --------------------------------------------------
# Save Silver
# --------------------------------------------------
products_clean.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.greenhouse_silver.products_clean"
)

shipments_clean.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.greenhouse_silver.shipments_clean"
)

shipments_quarantine.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.greenhouse_silver.shipments_quarantine"
)

receipts_clean.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.greenhouse_silver.receipts_clean"
)

receipts_quarantine.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.greenhouse_silver.receipts_quarantine"
)

# --------------------------------------------------
# Verify
# --------------------------------------------------
print("Valid shipments:", shipments_clean.count())
print("Rejected shipments:", shipments_quarantine.count())

print("Valid receipts:", receipts_clean.count())
print("Rejected receipts:", receipts_quarantine.count())

display(
    shipments_quarantine
    .groupBy("validation_reason")
    .count()
)

display(
    receipts_quarantine
    .groupBy("validation_reason")
    .count()
)

Products: 5
Shipments: 503
Receipts: 473
Bronze products: 5
Bronze shipments: 503
Bronze receipts: 473
Valid shipments: 499
Rejected shipments: 4
Valid receipts: 468
Rejected receipts: 5


validation_reason,count
MISSING_SHIPMENT_ID,1
DUPLICATE_SHIPMENT_ID,2
INVALID_SHIPPED_KG,1


validation_reason,count
MISSING_RECEIPT_ID,1
ORPHAN_SHIPMENT_ID,1
DUPLICATE_RECEIPT_ID,2
INVALID_RECEIVED_KG,1


In [0]:
%sql
CREATE OR REPLACE TABLE
workspace.greenhouse_gold.fact_shipment_reconciliation
AS

SELECT
    s.shipment_id,
    s.product_id,
    p.product_name,
    s.shipment_date,
    s.shipped_kg,

    r.receipt_id,
    r.receipt_date,
    r.received_kg,

    CASE
        WHEN r.receipt_id IS NULL
            THEN 'PENDING_RECEIPT'

        WHEN r.received_kg = s.shipped_kg
            THEN 'MATCHED'

        ELSE 'QUANTITY_DISCREPANCY'
    END AS reconciliation_status,

    CASE
        WHEN r.received_kg IS NULL
            THEN NULL
        ELSE s.shipped_kg - r.received_kg
    END AS quantity_difference_kg,

    CASE
        WHEN r.received_kg IS NULL
            THEN NULL

        WHEN s.shipped_kg > r.received_kg
            THEN s.shipped_kg - r.received_kg

        ELSE 0
    END AS shortfall_kg

FROM workspace.greenhouse_silver.shipments_clean s

LEFT JOIN workspace.greenhouse_silver.receipts_clean r
    ON s.shipment_id = r.shipment_id

LEFT JOIN workspace.greenhouse_silver.products_clean p
    ON s.product_id = p.product_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    reconciliation_status,
    COUNT(*) AS record_count,
    ROUND(SUM(COALESCE(shortfall_kg, 0)), 2) AS total_shortfall_kg
FROM workspace.greenhouse_gold.fact_shipment_reconciliation
GROUP BY reconciliation_status
ORDER BY reconciliation_status;

reconciliation_status,record_count,total_shortfall_kg
MATCHED,452,0.0
PENDING_RECEIPT,31,0.0
QUANTITY_DISCREPANCY,16,235.0


In [0]:
%sql
CREATE OR REPLACE VIEW
workspace.greenhouse_gold.vw_reconciliation_kpis
AS

SELECT
    COUNT(*) AS total_shipments,

    SUM(
        CASE WHEN reconciliation_status = 'MATCHED'
        THEN 1 ELSE 0 END
    ) AS matched_shipments,

    SUM(
        CASE WHEN reconciliation_status = 'PENDING_RECEIPT'
        THEN 1 ELSE 0 END
    ) AS pending_receipts,

    SUM(
        CASE WHEN reconciliation_status = 'QUANTITY_DISCREPANCY'
        THEN 1 ELSE 0 END
    ) AS quantity_discrepancies,

    ROUND(
        SUM(COALESCE(shortfall_kg, 0)),
        2
    ) AS total_shortfall_kg

FROM workspace.greenhouse_gold.fact_shipment_reconciliation;

In [0]:
%sql
CREATE OR REPLACE VIEW
workspace.greenhouse_gold.vw_operational_data_quality
AS

SELECT
    'Shipments' AS dataset,
    503 AS bronze_records,
    499 AS accepted_records,
    4 AS rejected_records

UNION ALL

SELECT
    'Receipts',
    473,
    468,
    5;

In [0]:
%sql
SELECT *
FROM workspace.greenhouse_gold.vw_reconciliation_kpis;

SELECT *
FROM workspace.greenhouse_gold.vw_operational_data_quality;

dataset,bronze_records,accepted_records,rejected_records
Shipments,503,499,4
Receipts,473,468,5


In [0]:
%sql
SELECT *
FROM workspace.greenhouse_gold.vw_reconciliation_kpis;

total_shipments,matched_shipments,pending_receipts,quantity_discrepancies,total_shortfall_kg
499,452,31,16,235.0
